# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`mbse/Expressions/Conformance/Corpus.py`, mirrored statement for
statement in TypeScript) and commits its snapshots under `conformance/<implementation>/`. This suite checks this
implementation's files are current, that its JSON is byte-identical to every other implementation's, and that it reads
every other implementation's JSON and YAML back to the same terms. Regenerate with
`uv run python -m mbse.Expressions.Conformance.write`. The corpus has a case for Patterns and for every dialect, each a
term written with its dialect, then its kind; each case validates in its own dialect, within a holder in the
corpus's store, after `Patterns.define` and each dialect's `define`, C's and C++'s within a holder naming the ABI
their types are read under, x86-64's System V ABI. The cases that are data, not terms, are dialects'
singletons, each written as one document: `domains`, ISO/IEC 11404's dialect, its domains, owned by name, each kind its
dialect's then its own, an enum and a packed one among them; and each dialect's stored domains
(`systemverilog_domains`, `python3_domains`, `typescript5_domains`, `matlab_domains`, `excel_domains`, `c_domains`,
`cpp_domains`), and each ABI dialect's (`amd64_domains`, `aapcs64_domains`); their domains validate. One is invalid syntax, `invalid`: a Python3 term holding an `error`, a `missing` term and literals with their
spelling, one of them holding no value but its problem; its problems are reported.

In [ ]:
from pathlib import Path

import yaml

from mbse.Expressions.Conformance.Corpus import CASES, DIALECTS, DOMAINS, build
from mbse.Expressions.Conformance.write import render
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Patterns.Expressions import Expressions as E
from mbse.Expressions.Framework import Domains as FD, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import JSON, Plain, Validators, YAML

ROOT = Path("../../conformance")
OWN = "python3"
OTHERS = sorted(p.name for p in ROOT.iterdir() if p.is_dir() and p.name != OWN)
assert OTHERS == ["typescript5"], OTHERS

corpus = build()
snapshots = {case: Plain.ToPlain(store)(schema, root) for case, (schema, root, store) in corpus.items()}

<a id="CONF-01"></a>
## CONF-01 · This implementation's committed files are current

In [ ]:
for name, text in render(corpus).items():
    assert (ROOT / OWN / name).read_text(encoding="utf-8") == text, f"{name} is stale: regenerate the corpus"

<a id="CONF-02"></a>
## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
for name in [OWN, *OTHERS]:
    assert sorted(p.name for p in (ROOT / name).iterdir()) == sorted(f"{c}.{e}" for c in CASES for e in ("json", "yaml")), name
for other in OTHERS:
    for case in CASES:
        mine = (ROOT / OWN / f"{case}.json").read_bytes()
        theirs = (ROOT / other / f"{case}.json").read_bytes()
        assert mine == theirs, f"{case}.json differs from {other}"

<a id="CONF-03"></a>
## CONF-03 · Every implementation's YAML reads back to exactly this implementation's snapshot, also under YAML 1.1

In [ ]:
for other in [OWN, *OTHERS]:
    for case in CASES:
        text = (ROOT / other / f"{case}.yaml").read_text(encoding="utf-8")
        assert YAML.loads(text) == snapshots[case], f"{other}/{case}.yaml"
        assert yaml.safe_load(text) == snapshots[case], f"{other}/{case}.yaml under YAML 1.1"

<a id="CONF-04"></a>
## CONF-04 · Every implementation's JSON and YAML deserialize into this implementation's store, and validate

In [ ]:
ISO, SV = "isoiec11404", "systemverilog"
STORED = {  # each case that is data: its dialect's domains, in order, by name, each its kind, its dialect's tag then its own
    "domains": {"boolean": (ISO, "boolean"), "integer": (ISO, "integer"), "octetstring": (ISO, "octetstring"),
                "octet": (ISO, "octet"), "modulo(256)": (ISO, "modulo"), "state": (ISO, "state"), "mode": (ISO, "packed")},
    "systemverilog_domains": {**{name: (SV, "logic") for name in ("logic", "reg", "integer", "time")},
                              **{name: (SV, "bit") for name in ("bit", "byte", "shortint", "int", "longint")},
                              "real": ("ieee754", "ieee754"), "shortreal": ("ieee754", "ieee754")},
    "python3_domains": {"bool": (ISO, "boolean"), "int": (ISO, "integer"), "float": ("ieee754", "ieee754"),
                        "str": ("python3", "str"), "bytes": (ISO, "octetstring"), "NoneType": ("python3", "nonetype"),
                        "ellipsis": ("python3", "ellipsis")},
    "typescript5_domains": {"boolean": (ISO, "boolean"), "number": ("ieee754", "ieee754"), "bigint": (ISO, "integer"),
                            "string": ("typescript5", "string"), "null": ("typescript5", "null")},
    "matlab_domains": {"double": ("ieee754", "ieee754"), "logical": (ISO, "boolean"), "string": ("matlab", "string")},
    "excel_domains": {"number": ("ieee754", "ieee754"), "logical": (ISO, "boolean"), "text": ("excel", "text"),
                      "error": ("excel", "error")},
    "c_domains": {"bool": (ISO, "boolean"), **{f"int{w}_t": (ISO, "integer") for w in (8, 16, 32, 64)},
                  **{f"uint{w}_t": (ISO, "modulo") for w in (8, 16, 32, 64)},
                  **{name: ("ieee754", "ieee754") for name in ("_Float16", "_Float32", "_Float64", "_Float128", "_Decimal32",
                                                               "_Decimal64", "_Decimal128")}, "nullptr_t": ("c", "nullptr_t")},
    "cpp_domains": {"bool": (ISO, "boolean"), **{f"int{w}_t": (ISO, "integer") for w in (8, 16, 32, 64)},
                    **{f"uint{w}_t": (ISO, "modulo") for w in (8, 16, 32, 64)},
                    **{f"std::{name}": ("ieee754", "ieee754") for name in ("float16_t", "float32_t", "float64_t", "float128_t",
                                                                           "bfloat16_t")}, "nullptr_t": ("cpp", "nullptr_t")},
    **{case: {**{name: (ISO, kind) for name, kind in [  # each ABI's, as it fixes them: char signed or not, wchar_t too
        ("char", char), ("signed char", "integer"), ("unsigned char", "modulo"), ("short", "integer"),
        ("unsigned short", "modulo"), ("int", "integer"), ("unsigned int", "modulo"), ("long", "integer"),
        ("unsigned long", "modulo"), ("long long", "integer"), ("unsigned long long", "modulo")]},
        **{name: ("ieee754", "ieee754") for name in ("float", "double", "long double")},
        "wchar_t": (ISO, wchar), **{name: (ISO, "modulo") for name in ("char8_t", "char16_t", "char32_t", "size_t")},
        **{name: (ISO, "integer") for name in ("ptrdiff_t", "intptr_t")}, "uintptr_t": (ISO, "modulo"),
        **{f"int_least{w}_t": (ISO, "integer") for w in (8, 16, 32, 64)},
        **{f"uint_least{w}_t": (ISO, "modulo") for w in (8, 16, 32, 64)},
        "_Float32x": ("ieee754", "ieee754"), "_Float64x": ("ieee754", "ieee754")}
       for case, char, wchar in [("amd64_domains", "integer", "integer"), ("aapcs64_domains", "modulo", "modulo")]},
}
assert list(STORED) == list(DOMAINS)
for store in {id(store): store for _, _, store in corpus.values()}.values():
    store.transaction(Patterns.define)  # Patterns' operations, which validation reads from a term's holder's store
for other in [OWN, *OTHERS]:
    for case, (schema, root, store) in corpus.items():
        dialect = DIALECTS.get(case, E.DIALECT)
        for suffix, decode in [("json", JSON.FromJSON(store)), ("yaml", YAML.FromYAML(store))]:
            restored = decode(schema, (ROOT / other / f"{case}.{suffix}").read_bytes())
            assert Plain.ToPlain(store)(schema, restored) == snapshots[case], f"{other}/{case}.{suffix}"
            if case in DOMAINS:  # a dialect, a singleton: read back into itself, its domains valid
                assert restored is root and Validators.Validate(store)(schema, restored) == []
                assert [str(e.name) for e in restored.domains] == list(STORED[case]) and list(restored.operations) == []
                assert [(FD.name_of(e.domain), FD.kind_of(e.domain)[:2]) for e in restored.domains] == list(STORED[case].items())
                continue
            assert F.same(restored, root), f"{other}/{case}.{suffix}"
            with store.transaction():  # validated within a holder, in the corpus's store; C's and C++'s naming their ABI
                held = F.term(store.Expressions.Framework.Expression(lambda e: e.root(F.spec(restored)).abi(
                    [store.Instances.Expressions.SystemV.AMD64]) if case in ("c", "cpp") else e.root(F.spec(restored))))
            problems = dialect.validate(held, bound=SY.free(held, dialect))
            if case == "invalid":  # invalid syntax and a literal's problem, kept and reported
                assert problems == ["left: left: right: invalid syntax '(x +* 1)'",
                                    "left: right: leading zeros in a decimal integer literal are not permitted",
                                    "right: missing expression"], (other, problems)
                continue
            assert problems == [], (other, case, suffix)